In [2]:
# Добавляем папку src в PYTHONPATH из ноутбука
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent  # при необходимости скорректируй уровень
src_path = project_root / "src"

if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("src in path:", src_path)


src in path: D:\Projects\ROS_2\src\multi-drone-core\src


In [6]:
import numpy as np
from multi_drone_core.controllers.position_transformer import DroneLocalityState

def print_state(state, label):
    print(f"\n=== {label} ===")
    print("local_NED position:", state.get_position("local_NED"))
    print("local_ENU position:", state.get_position("local_ENU"))
    print("global_ENU position:", state.get_position("global_ENU"))
    print("global_NED position:", state.get_position("global_NED"))
    
    print("local_NED velocity:", state.get_velocity("local_NED"))
    print("local_ENU velocity:", state.get_velocity("local_ENU"))
    print("global_ENU velocity:", state.get_velocity("global_ENU"))
    print("global_NED velocity:", state.get_velocity("global_NED"))

    print("local_ENU euler:", state.get_orientation_euler("local_ENU"))
    print("local_NED euler:", state.get_orientation_euler("local_NED"))
    print("global_ENU euler:", state.get_orientation_euler("global_ENU"))
    print("global_NED euler:", state.get_orientation_euler("global_NED"))

    print("local_ENU quat:", state.get_orientation_quaternion("local_ENU"))
    print("local_NED quat:", state.get_orientation_quaternion("local_NED"))

state = DroneLocalityState()

# 1) Обновляем позицию в local_NED
state.update_position(np.array([1.0, 2.0, -3.0]), system="local_NED")
print_state(state, "After local_NED position update")

# 2) Обновляем скорость в local_ENU
state.update_velocity(np.array([4.0, -1.0, 2.5]), system="local_ENU")
print_state(state, "After local_ENU velocity update")

# 3) Обновляем ориентацию в local_ENU (yaw = +90°)
yaw = np.pi / 2.0
state.update_orientation_euler(np.array([0.0, 0.0, yaw]), system="local_ENU")
print_state(state, "After local_ENU yaw = +90deg")

# 4) Обновляем ориентацию в local_NED (yaw = -90°)
state.update_orientation_euler(np.array([0.0, 0.0, -yaw]), system="local_NED")
print_state(state, "After local_NED yaw = -90deg")



=== After local_NED position update ===
local_NED position: [ 1.  2. -3.]
local_ENU position: [2. 1. 3.]
global_ENU position: [2. 1. 3.]
global_NED position: [ 1.  2. -3.]
local_NED velocity: [0. 0. 0.]
local_ENU velocity: [0. 0. 0.]
global_ENU velocity: [0. 0. 0.]
global_NED velocity: [0. 0. 0.]
local_ENU euler: [0. 0. 0.]
local_NED euler: [0. 0. 0.]
global_ENU euler: [0. 0. 0.]
global_NED euler: [0. 0. 0.]
local_ENU quat: [0. 0. 0. 1.]
local_NED quat: [0. 0. 0. 1.]

=== After local_ENU velocity update ===
local_NED position: [ 1.  2. -3.]
local_ENU position: [2. 1. 3.]
global_ENU position: [2. 1. 3.]
global_NED position: [ 1.  2. -3.]
local_NED velocity: [-1.   4.  -2.5]
local_ENU velocity: [ 4.  -1.   2.5]
global_ENU velocity: [ 4.  -1.   2.5]
global_NED velocity: [-1.   4.  -2.5]
local_ENU euler: [0. 0. 0.]
local_NED euler: [0. 0. 0.]
global_ENU euler: [0. 0. 0.]
global_NED euler: [0. 0. 0.]
local_ENU quat: [0. 0. 0. 1.]
local_NED quat: [0. 0. 0. 1.]

=== After local_ENU yaw = +90

In [7]:
import numpy as np
from multi_drone_core.controllers.base_data import PositionData, OrientationData
from multi_drone_core.controllers.position_transformer import DroneLocalityState

def print_positions(state, label):
    print(f"\n=== {label} ===")
    print("local_ENU :", state.get_position("local_ENU"))
    print("local_NED :", state.get_position("local_NED"))
    print("global_ENU:", state.get_position("global_ENU"))
    print("global_NED:", state.get_position("global_NED"))

# мир сдвинут на +[10, 0, 0] в ENU
world_pos = PositionData(x=10.0, y=0.0, z=0.0)
world_ori = OrientationData(roll=0.0, pitch=0.0, yaw=0.0)

state = DroneLocalityState(world_position=world_pos, world_orientation=world_ori)

# задаём позицию дрона в локальной системе ENU
state.update_position(np.array([1.0, 2.0, -3.0]), system="local_ENU")
print_positions(state, "World offset only (position)")

# ожидаемо:
# global_ENU = local_ENU + world_pos = [11, 2, -3]
# local_NED = [2, 1, 3]
# global_NED = rotate_ENU_NED(global_ENU)



=== World offset only (position) ===
local_ENU : [ 1.  2. -3.]
local_NED : [2. 1. 3.]
global_ENU: [11.  2. -3.]
global_NED: [ 2. 11.  3.]


In [5]:
import numpy as np
from multi_drone_core.controllers.base_data import PositionData, OrientationData
from multi_drone_core.controllers.position_transformer import DroneLocalityState

def print_all(state, label):
    print(f"\n=== {label} ===")
    print("local_ENU pos :", state.get_position("local_ENU"))
    print("global_ENU pos:", state.get_position("global_ENU"))
    print("local_ENU euler:", state.get_orientation_euler("local_ENU"))
    print("global_ENU euler:", state.get_orientation_euler("global_ENU"))
    print("local_NED euler:", state.get_orientation_euler("local_NED"))
    print("global_NED euler:", state.get_orientation_euler("global_NED"))

# мир сдвинут и повернут
world_pos = PositionData(x=10.0, y=0.0, z=0.0)
world_ori = OrientationData(roll=0.0, pitch=0.0, yaw=np.pi/2)  # +90° вокруг Z

state = DroneLocalityState(world_position=world_pos, world_orientation=world_ori)

# 1) задаём локальную позицию
state.update_position(np.array([1.0, 0.0, 0.0]), system="local_ENU")

# 2) задаём локальную ориентацию (yaw = +30°)
state.update_orientation_euler(np.array([0.0, 0.0, np.pi/6]), system="local_ENU")

print_all(state, "World offset + world yaw + local yaw")

# ожидаемо:
# global_ENU position = R(world_yaw) * local_ENU + world_pos
# global_ENU yaw = world_yaw + local_yaw



=== World offset + world yaw + local yaw ===
local_ENU pos : [1. 0. 0.]
global_ENU pos: [10.  1.  0.]
local_ENU euler: [0.         0.         0.52359878]
global_ENU euler: [0.        0.        2.0943951]
local_NED euler: [3.14159265 0.         1.04719755]
global_NED euler: [ 3.14159265  0.         -0.52359878]
